# Thành viên 1
Các bảng phụ trách: `promotions.csv`, `inventory.csv`, `order_items.csv`, `order_item_promotions.csv`

Chạy lần lượt từ **Ô 0** đến ô cuối. Mỗi ô tạo đúng một file CSV, đọc từ dữ liệu gốc `student_data`.

## Ô 0: Chuẩn bị

In [1]:
from pathlib import Path
import zipfile
import pandas as pd

DATA = Path("/content/student_data")   # dữ liệu gốc (bronze)
OUT = Path("/content/silver")          # kết quả (silver)
OUT.mkdir(exist_ok=True)

if not DATA.exists():                  # chưa có dữ liệu -> chọn file student_data.zip
    from google.colab import files
    for name in files.upload():
        zipfile.ZipFile(name).extractall("/content")

def strip(df, cols):
    """Bỏ khoảng trắng thừa ở các cột chữ."""
    for c in cols:
        df[c] = df[c].astype("string").str.strip()
    return df

def save(df, name, pk):
    """Kiểm tra khóa chính (duy nhất, không rỗng) rồi ghi CSV."""
    pk = [pk] if isinstance(pk, str) else pk
    assert df[pk].notna().all().all() and not df.duplicated(pk).any(), f"{name}: khóa chính lỗi"
    df.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name}.csv: {len(df):,} dòng, khóa chính {pk} hợp lệ")
    return df.head()

Saving student_data.zip to student_data.zip


## Ô 1: `promotions.csv`

In [2]:
# Sửa: applicable_category để trống (NULL) thay vì chuỗi 'All' (trống = áp dụng mọi danh mục)
promotions = pd.read_csv(DATA / "promotions.csv")
strip(promotions, ["promo_id", "promo_name", "promo_type", "promo_channel", "applicable_category"])
for c in ["start_date", "end_date"]:
    promotions[c] = pd.to_datetime(promotions[c])

save(promotions.drop_duplicates("promo_id"), "promotions", "promo_id")

promotions.csv: 50 dòng, khóa chính ['promo_id'] hợp lệ


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,<NA>,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,<NA>,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,<NA>,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,<NA>,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


## Ô 2: `inventory.csv`

In [3]:
# Sửa: bỏ cột dư. product_name, category, segment đã có ở products; year, month suy ra từ snapshot_date
inventory = pd.read_csv(DATA / "inventory.csv").drop(columns=["product_name", "category", "segment", "year", "month"])
inventory["snapshot_date"] = pd.to_datetime(inventory["snapshot_date"])
assert (inventory[["stock_on_hand", "units_sold"]] >= 0).all().all(), "tồn kho hoặc số bán bị âm"

save(inventory, "inventory", ["snapshot_date", "product_id"])

inventory.csv: 60,247 dòng, khóa chính ['snapshot_date', 'product_id'] hợp lệ


,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate
0,2022-10-31,1,3,1,1,2,90.0,0.9333,1,0,0,0.2500
1,2022-11-30,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500
2,2022-12-31,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500
3,2016-04-30,3,35,13,11,2,95.5,0.9333,1,1,0,0.2391
4,2016-05-31,3,36,11,10,1,108.0,0.9667,1,1,0,0.2174


## Ô 3: `order_items.csv`

In [4]:
# Sửa: (1) giữ đủ 714.669 dòng: 16 cặp (order_id, product_id) trùng nhưng khác số lượng/giá là dòng hàng thật
#      (2) thêm khóa chính order_item_id  (3) bỏ promo_id, promo_id_2 (tách sang bảng order_item_promotions)
items = pd.read_csv(DATA / "order_items.csv",
                    usecols=["order_id", "product_id", "quantity", "unit_price", "discount_amount"])
items.insert(0, "order_item_id", items.index + 1)          # 1, 2, 3, ... theo thứ tự dòng gốc
assert (items["quantity"] >= 1).all() and (items["unit_price"] >= 0).all()

save(items, "order_items", "order_item_id")

order_items.csv: 714,669 dòng, khóa chính ['order_item_id'] hợp lệ


,order_item_id,order_id,product_id,quantity,unit_price,discount_amount
0,1,1,2400,7,1138.22,0.0
1,2,2,609,7,10166.25,0.0
2,3,3,396,3,11220.33,0.0
3,4,4,635,5,10639.25,0.0
4,5,6,1935,1,1597.84,0.0


## Ô 4: `order_item_promotions.csv`

In [5]:
# Sửa: promo_id, promo_id_2 là nhóm lặp (vi phạm 1NF) -> tách thành bảng nối, mỗi dòng = 1 khuyến mãi của 1 dòng hàng
raw = pd.read_csv(DATA / "order_items.csv", usecols=["promo_id", "promo_id_2"], dtype="string")
raw.insert(0, "order_item_id", raw.index + 1)              # phải đánh số giống bảng order_items

promos = pd.concat([raw[["order_item_id", "promo_id"]],
                    raw[["order_item_id", "promo_id_2"]].rename(columns={"promo_id_2": "promo_id"})])
promos = strip(promos.dropna(), ["promo_id"]).drop_duplicates().sort_values(["order_item_id", "promo_id"])

save(promos.reset_index(drop=True), "order_item_promotions", ["order_item_id", "promo_id"])

/tmp/ipykernel_1663/2391414912.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = df[c].astype("string").str.strip()


order_item_promotions.csv: 276,522 dòng, khóa chính ['order_item_id', 'promo_id'] hợp lệ


,order_item_id,promo_id
0,41317,PROMO-0006
1,41318,PROMO-0006
2,41320,PROMO-0006
3,41321,PROMO-0006
4,41322,PROMO-0006
